# 11 · Architecture families & modern tricks

The same building blocks (04–10) get wired and optimised in different ways. This notebook is
conceptual — the map of the landscape.

## Architecture families
| Family | Attention | Sees | Good for | Examples |
|---|---|---|---|---|
| **Encoder-only** | bidirectional (no mask) | whole input at once | understanding, classification | BERT, RoBERTa |
| **Decoder-only** | causal (masked) | left context only | **generation** | GPT, Llama, Claude |
| **Encoder-decoder** | encoder reads, decoder writes | both | translation, summarization | T5, original Transformer |

Modern chat LLMs are almost all **decoder-only** — the causal mask (notebook 05) is the key
ingredient for left-to-right generation.

## Positional encoding variants (see notebook 03 for the original)
- **Learned absolute** (GPT-2): position vectors learned like embeddings. Simple, but caps max
  length.
- **RoPE — Rotary** (Llama, most modern models): *rotates* Q and K by a position-dependent
  angle. Encodes **relative** distance and extrapolates to longer contexts. Today's default.
- **ALiBi**: adds a distance penalty to attention scores. Cheap, long-context friendly.

How a model encodes "where" strongly affects its usable **context length**.

## Modern efficiency tricks
Vanilla attention is `O(seq_len²)` in time and memory. Production models add:

- **KV cache** — during generation, cache past Keys/Values so each new token is cheap instead
  of recomputing the whole sequence. The single biggest inference speedup.
- **GQA / MQA** (grouped / multi-query attention) — heads share Keys/Values → smaller KV cache,
  faster inference. Used in Llama-2/3.
- **FlashAttention** — computes exact attention without materializing the big `T×T` matrix;
  far less memory, much faster on GPU.
- **RMSNorm** — cheaper LayerNorm variant (skips mean-centering). Used by Llama.
- **SwiGLU** — a gated feed-forward that beats plain GELU. Common in modern models.
- **Mixture of Experts (MoE)** — many FFN "experts", route each token to a few → more
  parameters without more compute per token.

## Full recap
```
text ─► tokenize(01) ─► embed(02) + position(03)
     ─► [ LayerNorm ─► Multi-head Attention(04,05,06) ─► +residual
          LayerNorm ─► Feed-forward(07)               ─► +residual ] × N   (08,09)
     ─► LayerNorm ─► output head ─► softmax ─► next-token probability        (10)
```

Trained by next-token prediction, this gives a **Base** model. Then:
- **Base** — the raw pretrained Transformer above. Great autocomplete, can't follow orders.
- **Instruct** — *same architecture* + SFT + RLHF, so it follows instructions.
- **Reasoning** — *same architecture* + RL on math/code, so it thinks before answering.

**All three share this identical Transformer — they differ only in training, not
architecture.** (See `../base-instruct-reasoning-models.ipynb`.)

## Read next
- **The Illustrated Transformer** — Jay Alammar (best visual intro)
- **Attention Is All You Need** — Vaswani et al., 2017 (the paper)
- **The Annotated Transformer** — Harvard NLP (paper + line-by-line code)
- **nanoGPT** — Andrej Karpathy (a tiny trainable GPT — read every line)
- **3Blue1Brown** — "But what is a GPT?" (visual intuition)